# NB3: Compilación JIT con Numba

**Computación de Altas Prestaciones para Ciencia de Datos (CAPCD)**

---

## Objetivos de este notebook

1. Entender qué es la compilación **JIT (Just-In-Time)**.
2. Usar el decorador `@njit` para compilar funciones Python a código máquina.
3. Paralelizar bucles con `@njit(parallel=True)` y `prange`.
4. Conocer las limitaciones de Numba y cuándo usarlo.

In [ ]:
!pip install -q numba

---

## 1. ¿Qué es Numba?

**Numba** es un compilador JIT para Python que traduce funciones numéricas a código máquina optimizado usando **LLVM**.

### ¿Cómo funciona?

1. Escribes una función Python normal con bucles y operaciones numéricas.
2. Añades el decorador `@njit`.
3. La primera vez que llamas a la función, Numba la analiza, infiere tipos, y la compila a código máquina.
4. Las siguientes llamadas usan el código compilado directamente (velocidad de C).

### ¿Cuándo usar Numba?

| Numba funciona bien con | Numba NO funciona con |
|---|---|
| Bucles `for` con aritmética | Diccionarios complejos |
| Arrays NumPy | Pandas DataFrames |
| Operaciones matemáticas (`math`, `numpy`) | Strings, listas heterogéneas |
| Funciones que operan elemento a elemento | Llamadas a librerías externas (requests, etc.) |

---

## Demo 1: `@njit`, tu primer speedup de 100x

In [ ]:
import numpy as np
import time
from numba import njit


# --- Versión Python puro ---
def suma_cuadrados_python(data):
    """Suma de cuadrados en Python puro."""
    total = 0.0
    for i in range(len(data)):
        total += data[i] ** 2
    return total


# --- Versión Numba ---
@njit
def suma_cuadrados_numba(data):
    """Misma función, pero compilada con Numba."""
    total = 0.0
    for i in range(len(data)):
        total += data[i] ** 2
    return total


data = np.random.rand(10_000_000)

# Compilación (primera llamada)
print("Compilando Numba (primera llamada)...")
_ = suma_cuadrados_numba(data[:10])  # Warmup: compila con un array pequeño
print("Compilación completada.\n")

# Benchmark
start = time.perf_counter()
res_python = suma_cuadrados_python(data)
t_python = time.perf_counter() - start

start = time.perf_counter()
res_numba = suma_cuadrados_numba(data)
t_numba = time.perf_counter() - start

start = time.perf_counter()
res_numpy = np.sum(data ** 2)
t_numpy = time.perf_counter() - start

print(f"Python puro: {t_python:.4f}s")
print(f"Numba @njit: {t_numba:.4f}s  (speedup: {t_python/t_numba:.0f}x)")
print(f"NumPy:       {t_numpy:.4f}s  (speedup: {t_python/t_numpy:.0f}x)")
print(f"\n¿Resultados iguales? {np.isclose(res_python, res_numba) and np.isclose(res_python, res_numpy)}")

### Observaciones

- Numba compite con NumPy en velocidad, pero funciona con bucles `for` donde NumPy necesitaría vectorización compleja.
- La primera llamada es lenta (compilación). Las siguientes son instantáneas.
- El código Numba es idéntico al Python puro, solo se añade `@njit`.

---

## 2. El coste de compilación y warmup

La primera llamada a una función `@njit` es lenta porque Numba necesita:
1. Analizar el código Python.
2. Inferir los tipos de todas las variables.
3. Generar código LLVM IR.
4. Compilar a código máquina nativo.

Esto toma entre 0.1 y 2 segundos dependiendo de la complejidad.

### Buena práctica: warmup
```python
# Compila con datos pequeños antes del benchmark real
_ = mi_funcion_numba(datos_pequenos)
```

### Numba cachea: `@njit(cache=True)`
```python
@njit(cache=True)
def mi_funcion(x):
    ...
```
Guarda el código compilado en disco para reutilizarlo entre ejecuciones.

---

## Demo 2: Numba brilla con algoritmos imposibles de vectorizar

NumPy es rápido cuando puedes expresar todo con operaciones vectorizadas. Pero hay algoritmos que requieren bucles (dependencias entre iteraciones, condiciones complejas, etc.). Ahí es donde Numba gana.

In [ ]:
# Ejemplo: simulación de un random walk con barrera absorbente
# No se puede vectorizar fácilmente porque cada paso depende del anterior.

def random_walk_python(n_pasos, n_simulaciones):
    """Simula random walks y devuelve cuántos cruzan la barrera."""
    np.random.seed(42)
    cruzaron = 0
    barrera = 10.0
    for sim in range(n_simulaciones):
        posicion = 0.0
        for paso in range(n_pasos):
            posicion += np.random.randn()  # paso aleatorio
            if posicion > barrera:
                cruzaron += 1
                break  # barrera absorbente
    return cruzaron


@njit
def random_walk_numba(n_pasos, n_simulaciones):
    """Misma simulación, compilada con Numba."""
    np.random.seed(42)
    cruzaron = 0
    barrera = 10.0
    for sim in range(n_simulaciones):
        posicion = 0.0
        for paso in range(n_pasos):
            posicion += np.random.randn()
            if posicion > barrera:
                cruzaron += 1
                break
    return cruzaron


# Warmup
_ = random_walk_numba(10, 10)

N_PASOS = 1000
N_SIMS = 10_000

start = time.perf_counter()
res_python = random_walk_python(N_PASOS, N_SIMS)
t_python = time.perf_counter() - start

start = time.perf_counter()
res_numba = random_walk_numba(N_PASOS, N_SIMS)
t_numba = time.perf_counter() - start

print(f"Python: {t_python:.3f}s  ({res_python} cruzaron)")
print(f"Numba:  {t_numba:.3f}s  ({res_numba} cruzaron)")
print(f"Speedup: {t_python/t_numba:.0f}x")

---

## 3. Paralelización automática con `parallel=True` y `prange`

Numba puede **paralelizar bucles automáticamente** si las iteraciones son independientes.

```python
from numba import njit, prange

@njit(parallel=True)
def mi_funcion(data):
    resultado = np.zeros(len(data))
    for i in prange(len(data)):  # prange en lugar de range
        resultado[i] = data[i] ** 2
    return resultado
```

### Reglas para `prange`:
- Cada iteración debe ser independiente (no leer lo que otra escribió).
- Funciona bien con operaciones elemento a elemento sobre arrays.
- Numba detecta reducciones automáticamente (`total += ...`).

---

## Demo 3: `prange`, paralelismo sin procesos

In [ ]:
from numba import njit, prange
import numpy as np
import time


@njit
def distancia_euclidea_serial(A, B):
    """Calcula distancia euclídea entre pares de puntos (serial)."""
    n = A.shape[0]
    resultado = np.empty(n)
    for i in range(n):
        d = 0.0
        for j in range(A.shape[1]):
            d += (A[i, j] - B[i, j]) ** 2
        resultado[i] = np.sqrt(d)
    return resultado


@njit(parallel=True)
def distancia_euclidea_paralela(A, B):
    """Calcula distancia euclídea entre pares de puntos (paralelo)."""
    n = A.shape[0]
    resultado = np.empty(n)
    for i in prange(n):  # <-- prange en lugar de range
        d = 0.0
        for j in range(A.shape[1]):
            d += (A[i, j] - B[i, j]) ** 2
        resultado[i] = np.sqrt(d)
    return resultado


# Datos: 5M puntos en 50 dimensiones
N, D = 5_000_000, 50
A = np.random.rand(N, D)
B = np.random.rand(N, D)

# Warmup
_ = distancia_euclidea_serial(A[:10], B[:10])
_ = distancia_euclidea_paralela(A[:10], B[:10])

# Benchmark
start = time.perf_counter()
res_serial = distancia_euclidea_serial(A, B)
t_serial = time.perf_counter() - start

start = time.perf_counter()
res_paralela = distancia_euclidea_paralela(A, B)
t_paralela = time.perf_counter() - start

print(f"Numba serial:   {t_serial:.3f}s")
print(f"Numba paralelo: {t_paralela:.3f}s  (speedup: {t_serial/t_paralela:.1f}x)")
print(f"Resultados iguales: {np.allclose(res_serial, res_paralela)}")

### Ventaja sobre `ProcessPoolExecutor`

- **Sin overhead de serialización**: los datos no se copian entre procesos.
- **Sin GIL**: Numba libera el GIL internamente.
- **Granularidad fina**: paraleliza a nivel de iteración, no de función.
- **Más simple**: cambiar `range` → `prange` es todo lo que necesitas.

---

## Demo 4: Reducciones automáticas

Numba detecta automáticamente patrones de reducción (`total += ...`, `total *= ...`) y los paraleliza correctamente.

In [ ]:
@njit(parallel=True)
def suma_exp_cuad_paralela(matrix):
    """Suma de kernels gaussianos: sum(exp(-x²))
    Numba detecta 'total' como reducción y la paraleliza.
    Fusiona exp y la suma en un solo paso: sin arrays temporales.
    """
    total = 0.0
    for i in prange(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            x = matrix[i, j]
            total += np.exp(-x * x)  # Reducción automática
    return total

M = np.random.rand(5000, 5000)

# Warmup con array contiguo del mismo tipo
_ = suma_exp_cuad_paralela(np.random.rand(100, 100))

start = time.perf_counter()
res_numba = suma_exp_cuad_paralela(M)
t_numba = time.perf_counter() - start

# NumPy
start = time.perf_counter()
res_numpy = np.sum(np.exp(-M ** 2))
t_numpy = time.perf_counter() - start

print(f"Numba parallel: {t_numba:.4f}s")
print(f"NumPy:          {t_numpy:.4f}s")
print(f"Speedup:        {t_numpy/t_numba:.1f}x")
print(f"¿Iguales?: {np.isclose(res_numba, res_numpy)}")

---

## Ejercicio 1: Compilar con `@njit`

La siguiente función calcula la **distancia Manhattan** entre dos vectores usando Python puro. Crea una versión compilada con `@njit` y mide el speedup.

In [ ]:
import numpy as np
import time
from numba import njit


def manhattan_python(a, b):
    """Distancia Manhattan en Python puro."""
    total = 0.0
    for i in range(len(a)):
        total += abs(a[i] - b[i])
    return total


# TODO: Crea manhattan_numba usando @njit
# El código interior debe ser IDÉNTICO al de manhattan_python


# Test
a = np.random.rand(5_000_000)
b = np.random.rand(5_000_000)

# TODO: Haz warmup de manhattan_numba


start = time.perf_counter()
res_python = manhattan_python(a, b)
t_python = time.perf_counter() - start

start = time.perf_counter()
# res_numba = manhattan_numba(a, b)  # TODO: descomenta
t_numba = time.perf_counter() - start

print(f"Python: {t_python:.4f}s")
# print(f"Numba:  {t_numba:.4f}s  (speedup: {t_python/t_numba:.0f}x)")  # TODO: descomenta

In [ ]:
# Autoevaluación
# assert np.isclose(res_python, res_numba), "Los resultados deben coincidir"  # TODO: descomenta
# assert t_numba < t_python * 0.1, "Numba debería ser al menos 10x más rápido"  # TODO: descomenta
# print("✅ ¡Correcto! Numba acelera el cálculo drásticamente.")

---

## Ejercicio 2: Paralelizar con `prange`

Dada una matriz, calcula la **norma L2 de cada fila** (la raíz de la suma de cuadrados de cada fila). Usa `@njit(parallel=True)` y `prange`.

In [ ]:
from numba import njit, prange
import numpy as np
import time


@njit(parallel=True)
def normas_filas(matrix):
    """Calcula la norma L2 de cada fila de una matriz.

    Args:
        matrix: array 2D de forma (n_filas, n_cols)

    Returns:
        array 1D con la norma de cada fila
    """
    n_filas = matrix.shape[0]
    n_cols = matrix.shape[1]
    resultado = np.empty(n_filas)

    # TODO: Usa prange para el bucle exterior
    # Para cada fila, calcula sqrt(sum(matrix[i,j]^2 for j in cols))
    for i in range(n_filas):  # TODO: cambia range por prange
        pass  # TODO: implementa

    return resultado


# Test
M = np.random.rand(1_000_000, 100)

# Warmup
_ = normas_filas(M[:10])

start = time.perf_counter()
res_numba = normas_filas(M)
t_numba = time.perf_counter() - start

start = time.perf_counter()
res_numpy = np.linalg.norm(M, axis=1)
t_numpy = time.perf_counter() - start

print(f"Numba parallel: {t_numba:.3f}s")
print(f"NumPy norm:     {t_numpy:.3f}s")

In [ ]:
# Autoevaluación
assert np.allclose(res_numba, res_numpy, rtol=1e-5), "Los resultados deben coincidir con np.linalg.norm"
print("✅ ¡Correcto! Las normas calculadas con Numba coinciden con NumPy.")

---

## Ejercicio 3: Simulación Monte Carlo

Implementa una estimación de $\pi$ usando Monte Carlo con Numba:

1. Genera `n` puntos aleatorios $(x, y)$ en el cuadrado $[0, 1) \times [0, 1)$.
2. Cuenta cuántos caen dentro del círculo unitario: $x^2 + y^2 < 1$.
3. $\pi \approx 4 \times \frac{\text{dentro}}{n}$

Haz una versión serial y una paralela. Compara tiempos.

In [ ]:
from numba import njit, prange
import numpy as np
import time


@njit
def estimar_pi_serial(n):
    """Estima pi con Monte Carlo (serial)."""
    # TODO: Implementa
    # 1. Genera n puntos (x, y) con np.random.rand()
    # 2. Cuenta cuántos cumplen x**2 + y**2 < 1
    # 3. Devuelve 4 * dentro / n
    pass


@njit(parallel=True)
def estimar_pi_paralelo(n):
    """Estima pi con Monte Carlo (paralelo con prange)."""
    # TODO: Implementa usando prange
    # Pista: 'dentro' es una reducción (dentro += 1)
    pass


N = 50_000_000

# Warmup
# TODO: warmup de ambas funciones

# Benchmark
start = time.perf_counter()
pi_serial = estimar_pi_serial(N)
t_serial = time.perf_counter() - start

start = time.perf_counter()
pi_paralelo = estimar_pi_paralelo(N)
t_paralelo = time.perf_counter() - start

if pi_serial is not None and pi_paralelo is not None:
    print(f"Serial:   π ≈ {pi_serial:.6f}  ({t_serial:.3f}s)")
    print(f"Paralelo: π ≈ {pi_paralelo:.6f}  ({t_paralelo:.3f}s)")
    print(f"Real:     π = {np.pi:.6f}")
    print(f"Speedup: {t_serial/t_paralelo:.1f}x")
else:
    print("⚠️  Implementa estimar_pi_serial y estimar_pi_paralelo (elimina el 'pass').")

In [ ]:
# Autoevaluación
assert pi_serial is not None, "estimar_pi_serial no debe devolver None"
assert pi_paralelo is not None, "estimar_pi_paralelo no debe devolver None"
assert abs(pi_serial - np.pi) < 0.01, f"Estimación serial muy imprecisa: {pi_serial}"
assert abs(pi_paralelo - np.pi) < 0.01, f"Estimación paralela muy imprecisa: {pi_paralelo}"
print("✅ ¡Correcto! Ambas estimaciones son cercanas a π.")

---

## El futuro: JIT integrado en CPython

Desde **Python 3.13** (2024), CPython incluye un JIT experimental basado en la técnica *copy-and-patch*:

- **3.11**: introdujo un intérprete con especialización adaptativa de tipos (+10-25% velocidad).
- **3.12**: reestructuración interna para permitir modificar el intérprete en tiempo de compilación.
- **3.13**: detector de *hot spots* + JIT *copy-and-patch* que reemplaza bytecode por código máquina.

### ¿Sustituye a Numba?

**No.** El JIT de CPython es más ligero y general, pero no está diseñado para código numérico con NumPy:

| | JIT de CPython | Numba |
|---|---|---|
| **Objetivo** | Acelerar Python genérico | Acelerar bucles numéricos |
| **Técnica** | Copy-and-patch (stencils) | Compilación completa LLVM |
| **NumPy** | No afecta (ya es C interno) | Compila operaciones NumPy |
| **GPU** | No | Sí (`@cuda.jit`) |
| **Speedup típico** | Moderado (~1.1-1.5×) | Alto (~10-100× en bucles) |

El JIT de CPython hará que Python "base" sea más rápido, pero para ciencia de datos, Numba, CuPy y CUDA siguen siendo imprescindibles.

---

## Resumen

| Decorador | Efecto |
|---|---|
| `@njit` | Compila a código máquina (un núcleo) |
| `@njit(parallel=True)` | Compila y paraleliza bucles `prange` |
| `@njit(cache=True)` | Cachea la compilación entre ejecuciones |

### Cuándo usar Numba

- **Bucles numéricos** que no puedes vectorizar fácilmente con NumPy.
- **Simulaciones** (Monte Carlo, random walks, PDEs).
- **Procesamiento elemento a elemento** de grandes arrays.

### Siguiente paso

En los **NB4** y **NB5** veremos cómo llevar estos cálculos a la **GPU**, donde tenemos miles de núcleos en lugar de 2-8. Numba es el hilo conductor: pasaremos de `@njit` a `@cuda.jit`.